In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [2]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value[0])  # Ensure it returns a scalar

In [ ]:
min_qubits = 5
max_qubits=6
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                backend_options={ # method chosen automatically to match options
                    "coupling_map": coupling_map,
                },
                run_options={"seed": 42, "shots": 1024},
                transpile_options={"seed_transpiler": 42},
            )
    # Example usage
    import numpy as np

    from mealpy import FloatVar, BRO

    problem_dict = {
        "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
        "minmax": "min",
        "obj_func": evaluate_expectation
    }

    model = BRO.DevBRO(epoch=1000, pop_size=100, threshold = 3)
    g_best = model.solve(problem_dict)
    print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
    print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:08:22 PM, INFO, mealpy.human_based.BRO.DevBRO: Solving single objective optimization problem.


ansatz_num_parameters= 20


2024/07/11 01:08:23 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Problem: P, Epoch: 1, Current best: -1.5, Global best: -1.5625, Runtime: 0.54362 seconds
2024/07/11 01:08:23 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Problem: P, Epoch: 2, Current best: -1.5, Global best: -1.5625, Runtime: 0.53603 seconds
2024/07/11 01:08:24 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Problem: P, Epoch: 3, Current best: -1.53125, Global best: -1.5625, Runtime: 0.60060 seconds
2024/07/11 01:08:24 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Problem: P, Epoch: 4, Current best: -1.5625, Global best: -1.5625, Runtime: 0.50347 seconds
2024/07/11 01:08:25 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Problem: P, Epoch: 5, Current best: -1.5625, Global best: -1.5625, Runtime: 0.57979 seconds
2024/07/11 01:08:25 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Problem: P, Epoch: 6, Current best: -1.59375, Global best: -1.59375, Runtime: 0.57133 seconds
2024/07/11 01:08:26 PM, INFO, mealpy.human_based.BRO.DevBRO: >>>Pro

KeyboardInterrupt: 

In [7]:
from mealpy import FloatVar, BSO

problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = BSO.ImprovedBSO(epoch=1000, pop_size=50, m_clusters = 5, p1 = 0.25, p2 = 0.5, p3 = 0.75, p4 = 0.6)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:13:10 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: Solving single objective optimization problem.
2024/07/11 01:13:11 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: >>>Problem: P, Epoch: 1, Current best: -1.25, Global best: -1.25, Runtime: 0.14075 seconds
2024/07/11 01:13:11 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: >>>Problem: P, Epoch: 2, Current best: -1.15625, Global best: -1.25, Runtime: 0.14324 seconds
2024/07/11 01:13:11 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: >>>Problem: P, Epoch: 3, Current best: -1.15625, Global best: -1.25, Runtime: 0.14374 seconds
2024/07/11 01:13:11 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: >>>Problem: P, Epoch: 4, Current best: -1.09375, Global best: -1.25, Runtime: 0.13864 seconds
2024/07/11 01:13:11 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: >>>Problem: P, Epoch: 5, Current best: -1.09375, Global best: -1.25, Runtime: 0.14646 seconds
2024/07/11 01:13:11 PM, INFO, mealpy.human_based.BSO.ImprovedBSO: >>>Problem: P, Epoch: 6,

Solution: [-37.7191868    2.20675241 -12.67563092  15.09793034  -2.12353364
 -21.89589285  -3.72458248   3.11608599 -21.87156862 -42.16686795
 -14.99964656  14.06548132  -4.11828195  17.34410718  21.46355569
  29.75680991 -13.96323296 -23.77634999  20.5261365  -56.30011928], Fitness: -3.96875
Solution: [-37.7191868    2.20675241 -12.67563092  15.09793034  -2.12353364
 -21.89589285  -3.72458248   3.11608599 -21.87156862 -42.16686795
 -14.99964656  14.06548132  -4.11828195  17.34410718  21.46355569
  29.75680991 -13.96323296 -23.77634999  20.5261365  -56.30011928], Fitness: -3.96875


In [9]:
from mealpy import FloatVar, CA

problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = CA.OriginalCA(epoch=1000, pop_size=50, accepted_rate = 0.15)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:18:05 PM, INFO, mealpy.human_based.CA.OriginalCA: Solving single objective optimization problem.
2024/07/11 01:18:06 PM, INFO, mealpy.human_based.CA.OriginalCA: >>>Problem: P, Epoch: 1, Current best: -1.25, Global best: -1.25, Runtime: 0.16873 seconds
2024/07/11 01:18:06 PM, INFO, mealpy.human_based.CA.OriginalCA: >>>Problem: P, Epoch: 2, Current best: -1.25, Global best: -1.25, Runtime: 0.16873 seconds
2024/07/11 01:18:06 PM, INFO, mealpy.human_based.CA.OriginalCA: >>>Problem: P, Epoch: 3, Current best: -2.125, Global best: -2.125, Runtime: 0.16952 seconds
2024/07/11 01:18:06 PM, INFO, mealpy.human_based.CA.OriginalCA: >>>Problem: P, Epoch: 4, Current best: -2.125, Global best: -2.125, Runtime: 0.15034 seconds
2024/07/11 01:18:06 PM, INFO, mealpy.human_based.CA.OriginalCA: >>>Problem: P, Epoch: 5, Current best: -2.125, Global best: -2.125, Runtime: 0.14753 seconds
2024/07/11 01:18:07 PM, INFO, mealpy.human_based.CA.OriginalCA: >>>Problem: P, Epoch: 6, Current best: -2.12

KeyboardInterrupt: 

In [10]:
from mealpy import FloatVar, CHIO

problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = CHIO.OriginalCHIO(epoch=1000, pop_size=50, brr = 0.15, max_age = 100)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:19:11 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: Solving single objective optimization problem.
2024/07/11 01:19:12 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: >>>Problem: P, Epoch: 1, Current best: -1.25, Global best: -1.25, Runtime: 0.15591 seconds
2024/07/11 01:19:12 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: >>>Problem: P, Epoch: 2, Current best: -1.3125, Global best: -1.3125, Runtime: 0.15517 seconds
2024/07/11 01:19:12 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: >>>Problem: P, Epoch: 3, Current best: -1.3125, Global best: -1.3125, Runtime: 0.16075 seconds
2024/07/11 01:19:12 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: >>>Problem: P, Epoch: 4, Current best: -1.5, Global best: -1.5, Runtime: 0.17964 seconds
2024/07/11 01:19:12 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: >>>Problem: P, Epoch: 5, Current best: -1.5, Global best: -1.5, Runtime: 0.17659 seconds
2024/07/11 01:19:13 PM, INFO, mealpy.human_based.CHIO.OriginalCHIO: >>>Problem: P, Epo

Solution: [ -0.42841707  31.48932124  59.61025849 -23.66067171 -63.15740886
 -86.73773633 -21.32901475  28.81775288  33.08902991 -21.43758675
  10.63409555  57.52121007 -99.21037114  29.16175081   4.85826304
  77.51421807  10.50752202 -42.33701315  87.16963695 -99.45279046], Fitness: -3.34375
Solution: [ -0.42841707  31.48932124  59.61025849 -23.66067171 -63.15740886
 -86.73773633 -21.32901475  28.81775288  33.08902991 -21.43758675
  10.63409555  57.52121007 -99.21037114  29.16175081   4.85826304
  77.51421807  10.50752202 -42.33701315  87.16963695 -99.45279046], Fitness: -3.34375


In [16]:
import numpy as np
import time
from typing import Callable
from qiskit_algorithms.optimizers import OptimizerResult, OptimizerSupportLevel

class HyDEOptimizer:
    def __init__(self, I_NP=3000, F_weight=0.5, F_CR=0.6, I_itermax=3000, 
                 I_strategy=1, I_strategyVersion=1, I_bnd_constr=1, 
                 VarMin=-500, VarMax=500):
        self.I_NP = I_NP
        self.F_weight = F_weight
        self.F_CR = F_CR
        self.I_itermax = I_itermax
        self.I_strategy = I_strategy
        self.I_strategyVersion = I_strategyVersion
        self.I_bnd_constr = I_bnd_constr
        self.VarMin = VarMin
        self.VarMax = VarMax

    def get_support_level(self):
        return {
            "gradient": OptimizerSupportLevel.ignored,
            "bounds": OptimizerSupportLevel.required,
            "initial_point": OptimizerSupportLevel.required,
        }

    def minimize(self, fun: Callable, x0: np.ndarray) -> OptimizerResult:
        return self._hyde(fun, x0)

    def _hyde(self, fun, x0):
        start_time = time.time()
        de_parameters = {
            'I_NP': self.I_NP,
            'F_weight': self.F_weight,
            'F_CR': self.F_CR,
            'I_itermax': self.I_itermax,
            'I_strategy': self.I_strategy,
            'I_strategyVersion': self.I_strategyVersion,
            'I_bnd_constr': self.I_bnd_constr,
            'nVariables': len(x0),
            'minPositionsMatrix': np.tile(self.VarMin, (self.I_NP, len(x0))),
            'maxPositionsMatrix': np.tile(self.VarMax, (self.I_NP, len(x0)))
        }

        other_parameters = {
            'iRuns': np.random.randint(1000000),  # For reproducibility in the example, it can be adjusted
            'fnc': fun
        }

        low_habitat_limit = self.VarMin * np.ones(len(x0))
        up_habitat_limit = self.VarMax * np.ones(len(x0))

        # Initialization
        np.random.seed(other_parameters['iRuns'])
        FM_pop = self.genpop(de_parameters['I_NP'], len(x0), de_parameters['minPositionsMatrix'], de_parameters['maxPositionsMatrix'])

        S_val = np.array([fun(ind) for ind in FM_pop])
        I_best_index = np.argmin(S_val)
        FVr_bestmemit = FM_pop[I_best_index, :]
        fit_max_vector = np.zeros(de_parameters['I_itermax'])
        fit_max_vector[0] = S_val[I_best_index]

        FVr_rot = np.arange(de_parameters['I_NP'])
        F_weight_old, F_CR_old = None, None
        if de_parameters['I_strategy'] == 3:
            F_weight_old = np.tile(de_parameters['F_weight'], (de_parameters['I_NP'], 3))
            de_parameters['F_weight'] = F_weight_old
            F_CR_old = np.tile(de_parameters['F_CR'], (de_parameters['I_NP'], 1))
            de_parameters['F_CR'] = F_CR_old

        I_strategy_version = de_parameters['I_strategyVersion']
        other = {}

        gen = 1
        while gen < de_parameters['I_itermax']:
            other['a'] = (de_parameters['I_itermax'] - gen) / de_parameters['I_itermax']
            other['lowerlimit'] = low_habitat_limit
            other['upperlimit'] = up_habitat_limit

            if de_parameters['I_strategy'] == 3:
                value_R = np.random.rand(de_parameters['I_NP'], 3)
                ind1 = value_R < 0.1
                ind2 = np.random.rand(de_parameters['I_NP'], 1) < 0.1
                de_parameters['F_weight'][ind1] = 0.1 + np.random.rand(np.sum(ind1)) * 0.9
                de_parameters['F_weight'][~ind1] = F_weight_old[~ind1]
                de_parameters['F_CR'][ind2] = np.random.rand(np.sum(ind2))
                de_parameters['F_CR'][~ind2] = F_CR_old[~ind2]

            FM_ui, FM_base, _ = self.generate_trial(de_parameters['I_strategy'], de_parameters['F_weight'], de_parameters['F_CR'], FM_pop, FVr_bestmemit, de_parameters['I_NP'], len(x0), FVr_rot, I_strategy_version, other)
            FM_ui = self.update(FM_ui, de_parameters['minPositionsMatrix'], de_parameters['maxPositionsMatrix'], de_parameters['I_bnd_constr'], FM_base)
            S_val_temp = np.array([fun(ind) for ind in FM_ui])

            ind = np.where(S_val_temp < S_val)[0]
            S_val[ind] = S_val_temp[ind]
            FM_pop[ind, :] = FM_ui[ind, :]

            S_bestval = np.min(S_val)
            I_best_index = np.argmin(S_val)
            FVr_bestmemit = FM_pop[I_best_index, :]
            fit_max_vector[gen] = S_bestval

            if de_parameters['I_strategy'] == 3:
                F_weight_old[ind, :] = de_parameters['F_weight'][ind, :]
                F_CR_old[ind] = de_parameters['F_CR'][ind]

            print(f'Fitness value: {fit_max_vector[gen]}')
            print(f'Generation: {gen}')
            gen += 1

        result = OptimizerResult()
        result.x = FVr_bestmemit
        result.fun = fit_max_vector[gen-1]
        result.nfev = gen * de_parameters['I_NP']
        result.njev = None
        result.nit = gen
        result.time = time.time() - start_time

        return result

    def genpop(self, a, b, low_matrix, up_matrix):
        return np.random.uniform(low=low_matrix, high=up_matrix, size=(a, b))

    def update(self, p, low_matrix, up_matrix, BRM, FM_base):
        if BRM == 1:
            p = np.clip(p, low_matrix, up_matrix)
        elif BRM == 2:
            idx = np.where((p < low_matrix) | (p > up_matrix))
            p[idx] = np.random.uniform(low=low_matrix[idx], high=up_matrix[idx])
        elif BRM == 3:
            idx = np.where(p < low_matrix)
            p[idx] = np.random.uniform(low=low_matrix[idx], high=FM_base[idx])
            idx = np.where(p > up_matrix)
            p[idx] = np.random.uniform(low=FM_base[idx], high=up_matrix[idx])
        return p

    def generate_trial(self, method, F_weight, F_CR, FM_pop, FVr_bestmemit, I_NP, I_D, FVr_rot, I_strategy_version, other):
        FM_popold = FM_pop
        FVr_ind = np.random.permutation(4)
        FVr_a1 = np.random.permutation(I_NP)
        FVr_rt = (FVr_rot + FVr_ind[0]) % I_NP
        FVr_a2 = FVr_a1[FVr_rt]
        FVr_rt = (FVr_rot + FVr_ind[1]) % I_NP
        FVr_a3 = FVr_a2[FVr_rt]

        FM_pm1 = FM_popold[FVr_a1, :]
        FM_pm2 = FM_popold[FVr_a2, :]
        FM_pm3 = FM_popold[FVr_a3, :]

        FM_mui = np.random.rand(I_NP, I_D) < F_CR
        FM_mpo = FM_mui < 0.5
        if method == 1:
            FM_ui = FM_pm3 + F_weight * (FM_pm1 - FM_pm2)
            FM_ui = FM_popold * FM_mpo + FM_ui * FM_mui
            FM_base = FM_pm3
        elif method == 2:
            FM_bm = np.tile(FVr_bestmemit, (I_NP, 1))
            FM_ui = FM_popold + F_weight * (FM_bm - FM_popold) + F_weight * (FM_pm1 - FM_pm2)
            FM_ui = FM_popold * FM_mpo + FM_ui * FM_mui
            FM_base = FM_bm
        elif method == 3:
            FM_bm = np.tile(FVr_bestmemit, (I_NP, 1))
            if len(F_weight) == 1:
                FM_ui = FM_popold + F_weight * (FM_bm - FM_popold) + F_weight * (FM_pm1 - FM_pm2)
            else:
                if I_strategy_version == 1:
                    a = other['a']
                    ginv = np.log((1 + a) / a)
                    FM_ui = FM_popold + np.tile(F_weight[:, 1].reshape(-1, 1), (1, I_D)) * (FM_bm - FM_popold) + np.tile(F_weight[:, 0].reshape(-1, 1), (1, I_D)) * (FM_pm1 - FM_pm2) + np.tile(F_weight[:, 1].reshape(-1, 1), (1, I_D)) * ginv * (np.random.rand(I_NP, I_D) - 0.5)
                elif I_strategy_version == 2:
                    FM_ui = FM_popold + np.tile(F_weight[:, 1].reshape(-1, 1), (1, I_D)) * (FM_bm - FM_popold) + np.tile(F_weight[:, 0].reshape(-1, 1), (1, I_D)) * (FM_pm1 - FM_pm2)
            FM_ui = FM_popold * FM_mpo + FM_ui * FM_mui
            FM_base = FM_bm
        return FM_ui, FM_base, FM_mpo

def objective_function(x):
    # Example objective function: Rastrigin function
    return 100 * len(x) + np.sum(x**2 - 100 * np.cos(2 * np.pi * x))

optimizer = HyDEOptimizer(I_NP=300,I_itermax=1000 , I_strategy=1, I_strategyVersion=1)
x0 = np.random.uniform(-1000, 100, dimension)
result = optimizer.minimize(evaluate_expectation, x0)

print(result)


Fitness value: -1.5625
Generation: 1
Fitness value: -1.5625
Generation: 2
Fitness value: -2.21875
Generation: 3
Fitness value: -2.21875
Generation: 4
Fitness value: -2.21875
Generation: 5
Fitness value: -2.21875
Generation: 6
Fitness value: -2.21875
Generation: 7
Fitness value: -2.21875
Generation: 8
Fitness value: -2.21875
Generation: 9
Fitness value: -2.21875
Generation: 10
Fitness value: -2.75
Generation: 11
Fitness value: -2.75
Generation: 12
Fitness value: -2.75
Generation: 13
Fitness value: -2.75
Generation: 14
Fitness value: -2.75
Generation: 15
Fitness value: -2.9375
Generation: 16
Fitness value: -2.9375
Generation: 17
Fitness value: -2.9375
Generation: 18
Fitness value: -2.9375
Generation: 19
Fitness value: -2.9375
Generation: 20
Fitness value: -2.9375
Generation: 21
Fitness value: -2.9375
Generation: 22
Fitness value: -2.9375
Generation: 23
Fitness value: -2.9375
Generation: 24
Fitness value: -2.9375
Generation: 25
Fitness value: -2.9375
Generation: 26
Fitness value: -2.9375


KeyboardInterrupt: 

In [15]:
from scipy.optimize import differential_evolution
initial_point=2*np.pi*np.random.random(ansatz.num_parameters)-np.pi
bounds=[]
for j in range(0,ansatz.num_parameters):
    bounds.append((-np.pi,np.pi))
optimizer = differential_evolution(func=evaluate_expectation,strategy='best1bin',popsize=1,x0=initial_point,bounds=bounds,maxiter=20000,disp=True,init='halton',polish=False,tol=1e-2, updating='deferred')


differential_evolution step 1: f(x)= -1.21875
differential_evolution step 2: f(x)= -1.21875
differential_evolution step 3: f(x)= -1.21875
differential_evolution step 4: f(x)= -1.21875
differential_evolution step 5: f(x)= -1.53125
differential_evolution step 6: f(x)= -1.53125
differential_evolution step 7: f(x)= -1.53125
differential_evolution step 8: f(x)= -1.53125
differential_evolution step 9: f(x)= -1.53125
differential_evolution step 10: f(x)= -1.53125
differential_evolution step 11: f(x)= -1.53125
differential_evolution step 12: f(x)= -1.53125
differential_evolution step 13: f(x)= -1.78125
differential_evolution step 14: f(x)= -2.3125
differential_evolution step 15: f(x)= -2.3125
differential_evolution step 16: f(x)= -2.3125
differential_evolution step 17: f(x)= -2.3125
differential_evolution step 18: f(x)= -2.3125
differential_evolution step 19: f(x)= -2.3125
differential_evolution step 20: f(x)= -2.3125
differential_evolution step 21: f(x)= -2.3125
differential_evolution step 22

In [18]:
from mealpy import FloatVar, FBIO

problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = FBIO.DevFBIO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:45:45 PM, INFO, mealpy.human_based.FBIO.DevFBIO: Solving single objective optimization problem.
2024/07/11 01:45:46 PM, INFO, mealpy.human_based.FBIO.DevFBIO: >>>Problem: P, Epoch: 1, Current best: -1.8125, Global best: -1.8125, Runtime: 0.70034 seconds
2024/07/11 01:45:47 PM, INFO, mealpy.human_based.FBIO.DevFBIO: >>>Problem: P, Epoch: 2, Current best: -1.90625, Global best: -1.90625, Runtime: 0.64395 seconds
2024/07/11 01:45:47 PM, INFO, mealpy.human_based.FBIO.DevFBIO: >>>Problem: P, Epoch: 3, Current best: -1.90625, Global best: -1.90625, Runtime: 0.59155 seconds
2024/07/11 01:45:48 PM, INFO, mealpy.human_based.FBIO.DevFBIO: >>>Problem: P, Epoch: 4, Current best: -1.90625, Global best: -1.90625, Runtime: 0.66545 seconds
2024/07/11 01:45:49 PM, INFO, mealpy.human_based.FBIO.DevFBIO: >>>Problem: P, Epoch: 5, Current best: -2.125, Global best: -2.125, Runtime: 0.61416 seconds
2024/07/11 01:45:49 PM, INFO, mealpy.human_based.FBIO.DevFBIO: >>>Problem: P, Epoch: 6, Current 

KeyboardInterrupt: 

In [19]:
from mealpy import FloatVar, GSKA
    
problem_dict = {
    "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = GSKA.DevGSKA(epoch=1000, pop_size=50, pb = 0.1, kr = 0.9)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:47:47 PM, INFO, mealpy.human_based.GSKA.DevGSKA: Solving single objective optimization problem.
2024/07/11 01:47:48 PM, INFO, mealpy.human_based.GSKA.DevGSKA: >>>Problem: P, Epoch: 1, Current best: -1.25, Global best: -1.25, Runtime: 0.18695 seconds
2024/07/11 01:47:48 PM, INFO, mealpy.human_based.GSKA.DevGSKA: >>>Problem: P, Epoch: 2, Current best: -1.34375, Global best: -1.34375, Runtime: 0.27093 seconds
2024/07/11 01:47:48 PM, INFO, mealpy.human_based.GSKA.DevGSKA: >>>Problem: P, Epoch: 3, Current best: -1.46875, Global best: -1.46875, Runtime: 0.19852 seconds
2024/07/11 01:47:48 PM, INFO, mealpy.human_based.GSKA.DevGSKA: >>>Problem: P, Epoch: 4, Current best: -1.46875, Global best: -1.46875, Runtime: 0.27864 seconds
2024/07/11 01:47:48 PM, INFO, mealpy.human_based.GSKA.DevGSKA: >>>Problem: P, Epoch: 5, Current best: -1.46875, Global best: -1.46875, Runtime: 0.17517 seconds
2024/07/11 01:47:49 PM, INFO, mealpy.human_based.GSKA.DevGSKA: >>>Problem: P, Epoch: 6, Current 

Solution: [ 41.4230545   15.15624738 -37.05285819  25.00165162 -49.29172778
 -62.01022312 -48.98261324 -47.44402291 -20.91557987   7.71592246
 -69.75530805 -23.76402788 -63.10856424  55.11917307  98.63108113
 -17.86750319  42.87239085  14.857163    17.81662456  55.88163475], Fitness: -2.65625
Solution: [ 41.4230545   15.15624738 -37.05285819  25.00165162 -49.29172778
 -62.01022312 -48.98261324 -47.44402291 -20.91557987   7.71592246
 -69.75530805 -23.76402788 -63.10856424  55.11917307  98.63108113
 -17.86750319  42.87239085  14.857163    17.81662456  55.88163475], Fitness: -2.65625


In [20]:
objective_function = evaluate_expectation

In [22]:
from mealpy import FloatVar, HBO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = HBO.OriginalHBO(epoch=1000, pop_size=50, degree = 3)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:52:18 PM, INFO, mealpy.human_based.HBO.OriginalHBO: Solving single objective optimization problem.
2024/07/11 01:52:19 PM, INFO, mealpy.human_based.HBO.OriginalHBO: >>>Problem: P, Epoch: 1, Current best: -1.4375, Global best: -1.4375, Runtime: 0.18956 seconds
2024/07/11 01:52:19 PM, INFO, mealpy.human_based.HBO.OriginalHBO: >>>Problem: P, Epoch: 2, Current best: -1.4375, Global best: -1.4375, Runtime: 0.18249 seconds
2024/07/11 01:52:19 PM, INFO, mealpy.human_based.HBO.OriginalHBO: >>>Problem: P, Epoch: 3, Current best: -1.4375, Global best: -1.4375, Runtime: 0.17226 seconds
2024/07/11 01:52:19 PM, INFO, mealpy.human_based.HBO.OriginalHBO: >>>Problem: P, Epoch: 4, Current best: -1.4375, Global best: -1.4375, Runtime: 0.18543 seconds
2024/07/11 01:52:19 PM, INFO, mealpy.human_based.HBO.OriginalHBO: >>>Problem: P, Epoch: 5, Current best: -1.4375, Global best: -1.4375, Runtime: 0.18030 seconds
2024/07/11 01:52:19 PM, INFO, mealpy.human_based.HBO.OriginalHBO: >>>Problem: P, E

Solution: [ 50.40085374  35.52394879 -76.31324564  54.52466188 -91.58688265
 -14.52968899 -89.68088414 -44.72964212 -51.27393466 -41.88072789
 -95.72569822  12.87891804  67.50914767  58.59507225  54.94461947
  35.81781024  18.60292664 -53.79053759  -6.79949774 -46.63452783], Fitness: -3.21875
Solution: [ 50.40085374  35.52394879 -76.31324564  54.52466188 -91.58688265
 -14.52968899 -89.68088414 -44.72964212 -51.27393466 -41.88072789
 -95.72569822  12.87891804  67.50914767  58.59507225  54.94461947
  35.81781024  18.60292664 -53.79053759  -6.79949774 -46.63452783], Fitness: -3.21875


In [24]:
from mealpy import FloatVar, HCO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = HCO.OriginalHCO(epoch=1000, pop_size=50, wfp=0.65, wfv=0.05, c1=1.4, c2=1.4)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 01:56:18 PM, INFO, mealpy.human_based.HCO.OriginalHCO: Solving single objective optimization problem.
2024/07/11 01:56:19 PM, INFO, mealpy.human_based.HCO.OriginalHCO: >>>Problem: P, Epoch: 1, Current best: -1.5625, Global best: -1.5625, Runtime: 0.17466 seconds
2024/07/11 01:56:19 PM, INFO, mealpy.human_based.HCO.OriginalHCO: >>>Problem: P, Epoch: 2, Current best: -1.59375, Global best: -1.59375, Runtime: 0.17051 seconds
2024/07/11 01:56:19 PM, INFO, mealpy.human_based.HCO.OriginalHCO: >>>Problem: P, Epoch: 3, Current best: -1.59375, Global best: -1.59375, Runtime: 0.17251 seconds
2024/07/11 01:56:19 PM, INFO, mealpy.human_based.HCO.OriginalHCO: >>>Problem: P, Epoch: 4, Current best: -1.625, Global best: -1.625, Runtime: 0.16886 seconds
2024/07/11 01:56:19 PM, INFO, mealpy.human_based.HCO.OriginalHCO: >>>Problem: P, Epoch: 5, Current best: -1.65625, Global best: -1.65625, Runtime: 0.18357 seconds
2024/07/11 01:56:19 PM, INFO, mealpy.human_based.HCO.OriginalHCO: >>>Problem: 

TypeError: Invalid parameter values, expected Sequence[Sequence[float]].

In [26]:
from mealpy import FloatVar, ICA
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = ICA.OriginalICA(epoch=300, pop_size=50, empire_count = 5, assimilation_coeff = 1.5,
                        revolution_prob = 0.05, revolution_rate = 0.1, revolution_step_size = 0.1, zeta = 0.1)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 02:26:27 PM, INFO, mealpy.human_based.ICA.OriginalICA: Solving single objective optimization problem.
2024/07/11 02:26:27 PM, INFO, mealpy.human_based.ICA.OriginalICA: >>>Problem: P, Epoch: 1, Current best: -1.78125, Global best: -1.78125, Runtime: 0.19475 seconds
2024/07/11 02:26:27 PM, INFO, mealpy.human_based.ICA.OriginalICA: >>>Problem: P, Epoch: 2, Current best: -2.0, Global best: -2.0, Runtime: 0.18913 seconds
2024/07/11 02:26:27 PM, INFO, mealpy.human_based.ICA.OriginalICA: >>>Problem: P, Epoch: 3, Current best: -2.0, Global best: -2.0, Runtime: 0.18676 seconds
2024/07/11 02:26:28 PM, INFO, mealpy.human_based.ICA.OriginalICA: >>>Problem: P, Epoch: 4, Current best: -2.0, Global best: -2.0, Runtime: 0.20779 seconds
2024/07/11 02:26:28 PM, INFO, mealpy.human_based.ICA.OriginalICA: >>>Problem: P, Epoch: 5, Current best: -2.03125, Global best: -2.03125, Runtime: 0.23227 seconds
2024/07/11 02:26:28 PM, INFO, mealpy.human_based.ICA.OriginalICA: >>>Problem: P, Epoch: 6, Curre

Solution: [ 95.17820892  68.50499957  17.47751983  96.6080842   23.15751076
  15.22043049  70.56131879 -55.33722504 -41.83713604 -64.53100966
 -62.77201819 -73.77839217 -45.8037984   20.6355645   -8.02106856
 -95.46261233  95.96689064  -0.85189395  36.04000768 -46.69823831], Fitness: -3.96875
Solution: [ 95.17820892  68.50499957  17.47751983  96.6080842   23.15751076
  15.22043049  70.56131879 -55.33722504 -41.83713604 -64.53100966
 -62.77201819 -73.77839217 -45.8037984   20.6355645   -8.02106856
 -95.46261233  95.96689064  -0.85189395  36.04000768 -46.69823831], Fitness: -3.96875


In [27]:
from mealpy import FloatVar, LCO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = LCO.OriginalLCO(epoch=1000, pop_size=50, r1 = 2.35)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 02:29:40 PM, INFO, mealpy.human_based.LCO.OriginalLCO: Solving single objective optimization problem.
2024/07/11 02:29:40 PM, INFO, mealpy.human_based.LCO.OriginalLCO: >>>Problem: P, Epoch: 1, Current best: -1.8125, Global best: -1.8125, Runtime: 0.16198 seconds
2024/07/11 02:29:40 PM, INFO, mealpy.human_based.LCO.OriginalLCO: >>>Problem: P, Epoch: 2, Current best: -1.8125, Global best: -1.8125, Runtime: 0.15635 seconds
2024/07/11 02:29:40 PM, INFO, mealpy.human_based.LCO.OriginalLCO: >>>Problem: P, Epoch: 3, Current best: -1.8125, Global best: -1.8125, Runtime: 0.15886 seconds
2024/07/11 02:29:41 PM, INFO, mealpy.human_based.LCO.OriginalLCO: >>>Problem: P, Epoch: 4, Current best: -1.8125, Global best: -1.8125, Runtime: 0.15731 seconds
2024/07/11 02:29:41 PM, INFO, mealpy.human_based.LCO.OriginalLCO: >>>Problem: P, Epoch: 5, Current best: -1.8125, Global best: -1.8125, Runtime: 0.15637 seconds
2024/07/11 02:29:41 PM, INFO, mealpy.human_based.LCO.OriginalLCO: >>>Problem: P, E

Solution: [-13.80757057 -13.76728809 -13.7644516  -13.80176193 -13.82115407
 -13.81625898 -13.75757115 -13.8143228  -13.78345984 -13.80504642
 -13.78848515 -13.81587551 -13.75727344 -13.82019121 -13.81274273
 -13.81327893 -13.80111516 -13.82137176 -13.80074758 -13.78267921], Fitness: -3.9375
Solution: [-13.80757057 -13.76728809 -13.7644516  -13.80176193 -13.82115407
 -13.81625898 -13.75757115 -13.8143228  -13.78345984 -13.80504642
 -13.78848515 -13.81587551 -13.75727344 -13.82019121 -13.81274273
 -13.81327893 -13.80111516 -13.82137176 -13.80074758 -13.78267921], Fitness: -3.9375


In [28]:
from mealpy import FloatVar, QSA
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = QSA.DevQSA(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 02:32:17 PM, INFO, mealpy.human_based.QSA.DevQSA: Solving single objective optimization problem.
2024/07/11 02:32:17 PM, INFO, mealpy.human_based.QSA.DevQSA: >>>Problem: P, Epoch: 1, Current best: -1.21875, Global best: -1.21875, Runtime: 0.45683 seconds
2024/07/11 02:32:18 PM, INFO, mealpy.human_based.QSA.DevQSA: >>>Problem: P, Epoch: 2, Current best: -1.625, Global best: -1.625, Runtime: 0.43541 seconds
2024/07/11 02:32:18 PM, INFO, mealpy.human_based.QSA.DevQSA: >>>Problem: P, Epoch: 3, Current best: -1.625, Global best: -1.625, Runtime: 0.43576 seconds
2024/07/11 02:32:19 PM, INFO, mealpy.human_based.QSA.DevQSA: >>>Problem: P, Epoch: 4, Current best: -1.9375, Global best: -1.9375, Runtime: 0.43619 seconds
2024/07/11 02:32:19 PM, INFO, mealpy.human_based.QSA.DevQSA: >>>Problem: P, Epoch: 5, Current best: -1.9375, Global best: -1.9375, Runtime: 0.43551 seconds
2024/07/11 02:32:19 PM, INFO, mealpy.human_based.QSA.DevQSA: >>>Problem: P, Epoch: 6, Current best: -1.9375, Globa

Solution: [ -50.97630579  -96.69390676  -97.86899255  -77.96251717   52.4858573
   95.60232269  -86.67776799   12.57564629   11.29368389  -58.67454685
   64.21100061  -32.40673839  -76.46206521 -100.          -45.69141744
  -55.05443833   44.25516573  -48.82524278  -44.22797155  -67.42888121], Fitness: -3.78125
Solution: [ -50.97630579  -96.69390676  -97.86899255  -77.96251717   52.4858573
   95.60232269  -86.67776799   12.57564629   11.29368389  -58.67454685
   64.21100061  -32.40673839  -76.46206521 -100.          -45.69141744
  -55.05443833   44.25516573  -48.82524278  -44.22797155  -67.42888121], Fitness: -3.78125


In [30]:
from mealpy import FloatVar, SARO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = SARO.DevSARO(epoch=1000, pop_size=50, se = 0.5, mu = 15)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 02:40:26 PM, INFO, mealpy.human_based.SARO.DevSARO: Solving single objective optimization problem.
2024/07/11 02:40:26 PM, INFO, mealpy.human_based.SARO.DevSARO: >>>Problem: P, Epoch: 1, Current best: -1.75, Global best: -1.75, Runtime: 0.29602 seconds
2024/07/11 02:40:27 PM, INFO, mealpy.human_based.SARO.DevSARO: >>>Problem: P, Epoch: 2, Current best: -1.75, Global best: -1.75, Runtime: 0.29746 seconds
2024/07/11 02:40:27 PM, INFO, mealpy.human_based.SARO.DevSARO: >>>Problem: P, Epoch: 3, Current best: -1.75, Global best: -1.75, Runtime: 0.34663 seconds
2024/07/11 02:40:27 PM, INFO, mealpy.human_based.SARO.DevSARO: >>>Problem: P, Epoch: 4, Current best: -1.75, Global best: -1.75, Runtime: 0.31964 seconds
2024/07/11 02:40:28 PM, INFO, mealpy.human_based.SARO.DevSARO: >>>Problem: P, Epoch: 5, Current best: -1.75, Global best: -1.75, Runtime: 0.29765 seconds
2024/07/11 02:40:28 PM, INFO, mealpy.human_based.SARO.DevSARO: >>>Problem: P, Epoch: 6, Current best: -1.75, Global best

Solution: [-39.02750204  21.44017883 -69.71736298 -84.84715692 -25.74598751
  55.12629126  58.4458737  -21.71236942  37.51577321 -24.08969377
  23.43206057  14.09521548  83.43608311  57.35527939   5.00316683
  22.89523384 -95.97749837 -61.23675702  11.01406864 -83.30350243], Fitness: -3.96875
Solution: [-39.02750204  21.44017883 -69.71736298 -84.84715692 -25.74598751
  55.12629126  58.4458737  -21.71236942  37.51577321 -24.08969377
  23.43206057  14.09521548  83.43608311  57.35527939   5.00316683
  22.89523384 -95.97749837 -61.23675702  11.01406864 -83.30350243], Fitness: -3.96875


In [31]:
from mealpy import FloatVar, SPBO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = SPBO.OriginalSPBO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 02:45:17 PM, INFO, mealpy.human_based.SPBO.OriginalSPBO: Solving single objective optimization problem.
2024/07/11 02:45:20 PM, INFO, mealpy.human_based.SPBO.OriginalSPBO: >>>Problem: P, Epoch: 1, Current best: -2.125, Global best: -2.125, Runtime: 2.60061 seconds
2024/07/11 02:45:22 PM, INFO, mealpy.human_based.SPBO.OriginalSPBO: >>>Problem: P, Epoch: 2, Current best: -2.28125, Global best: -2.28125, Runtime: 2.65054 seconds
2024/07/11 02:45:25 PM, INFO, mealpy.human_based.SPBO.OriginalSPBO: >>>Problem: P, Epoch: 3, Current best: -2.28125, Global best: -2.28125, Runtime: 2.66203 seconds
2024/07/11 02:45:28 PM, INFO, mealpy.human_based.SPBO.OriginalSPBO: >>>Problem: P, Epoch: 4, Current best: -2.28125, Global best: -2.28125, Runtime: 2.68132 seconds
2024/07/11 02:45:30 PM, INFO, mealpy.human_based.SPBO.OriginalSPBO: >>>Problem: P, Epoch: 5, Current best: -2.28125, Global best: -2.28125, Runtime: 2.66174 seconds
2024/07/11 02:45:33 PM, INFO, mealpy.human_based.SPBO.OriginalSP

KeyboardInterrupt: 

In [36]:
from mealpy import FloatVar, SSDO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = SSDO.OriginalSSDO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 03:20:58 PM, INFO, mealpy.human_based.SSDO.OriginalSSDO: Solving single objective optimization problem.
2024/07/11 03:20:58 PM, INFO, mealpy.human_based.SSDO.OriginalSSDO: >>>Problem: P, Epoch: 1, Current best: -1.96875, Global best: -1.96875, Runtime: 0.17964 seconds
2024/07/11 03:20:58 PM, INFO, mealpy.human_based.SSDO.OriginalSSDO: >>>Problem: P, Epoch: 2, Current best: -1.96875, Global best: -1.96875, Runtime: 0.15588 seconds
2024/07/11 03:20:58 PM, INFO, mealpy.human_based.SSDO.OriginalSSDO: >>>Problem: P, Epoch: 3, Current best: -1.96875, Global best: -1.96875, Runtime: 0.14927 seconds
2024/07/11 03:20:59 PM, INFO, mealpy.human_based.SSDO.OriginalSSDO: >>>Problem: P, Epoch: 4, Current best: -1.96875, Global best: -1.96875, Runtime: 0.17532 seconds
2024/07/11 03:20:59 PM, INFO, mealpy.human_based.SSDO.OriginalSSDO: >>>Problem: P, Epoch: 5, Current best: -1.96875, Global best: -1.96875, Runtime: 0.18137 seconds
2024/07/11 03:20:59 PM, INFO, mealpy.human_based.SSDO.Origin

Solution: [-1.07029903e+00 -4.26954055e+01 -3.87549561e+01 -9.67668938e+01
 -3.58387075e+00  1.83562695e+01 -5.43093915e+01  1.07165757e+01
  3.25891808e+01  2.10467205e-01 -5.08849363e+00 -1.00000000e+02
  1.39782738e+00  2.46840939e+01 -1.84359116e+01 -6.16393490e+01
  1.00000000e+02 -4.77406264e+00 -8.00701519e-02 -6.92060827e+01], Fitness: -2.8125
Solution: [-1.07029903e+00 -4.26954055e+01 -3.87549561e+01 -9.67668938e+01
 -3.58387075e+00  1.83562695e+01 -5.43093915e+01  1.07165757e+01
  3.25891808e+01  2.10467205e-01 -5.08849363e+00 -1.00000000e+02
  1.39782738e+00  2.46840939e+01 -1.84359116e+01 -6.16393490e+01
  1.00000000e+02 -4.77406264e+00 -8.00701519e-02 -6.92060827e+01], Fitness: -2.8125


In [37]:
from mealpy import FloatVar, TLO
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = TLO.DevTLO(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 03:23:24 PM, INFO, mealpy.human_based.TLO.DevTLO: Solving single objective optimization problem.
2024/07/11 03:23:24 PM, INFO, mealpy.human_based.TLO.DevTLO: >>>Problem: P, Epoch: 1, Current best: -1.3125, Global best: -1.3125, Runtime: 0.28036 seconds
2024/07/11 03:23:24 PM, INFO, mealpy.human_based.TLO.DevTLO: >>>Problem: P, Epoch: 2, Current best: -1.46875, Global best: -1.46875, Runtime: 0.28243 seconds
2024/07/11 03:23:25 PM, INFO, mealpy.human_based.TLO.DevTLO: >>>Problem: P, Epoch: 3, Current best: -1.46875, Global best: -1.46875, Runtime: 0.28240 seconds
2024/07/11 03:23:25 PM, INFO, mealpy.human_based.TLO.DevTLO: >>>Problem: P, Epoch: 4, Current best: -1.46875, Global best: -1.46875, Runtime: 0.27738 seconds
2024/07/11 03:23:25 PM, INFO, mealpy.human_based.TLO.DevTLO: >>>Problem: P, Epoch: 5, Current best: -1.53125, Global best: -1.53125, Runtime: 0.28848 seconds
2024/07/11 03:23:25 PM, INFO, mealpy.human_based.TLO.DevTLO: >>>Problem: P, Epoch: 6, Current best: -1.5

Solution: [-12.41644354  -1.01980928 -84.05409706  58.85126928 -92.96818831
  22.61708577 -38.13085324  91.71137682  -8.90497051 -52.02671578
   2.08997647  -7.97802619  19.5036571  -96.12707967 -18.0742248
 -17.33693601 -52.09121384  -8.52652842  14.45688673  19.06604725], Fitness: -3.3125
Solution: [-12.41644354  -1.01980928 -84.05409706  58.85126928 -92.96818831
  22.61708577 -38.13085324  91.71137682  -8.90497051 -52.02671578
   2.08997647  -7.97802619  19.5036571  -96.12707967 -18.0742248
 -17.33693601 -52.09121384  -8.52652842  14.45688673  19.06604725], Fitness: -3.3125


In [38]:
from mealpy import FloatVar, TOA
    

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = TOA.OriginalTOA(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 03:28:17 PM, INFO, mealpy.human_based.TOA.OriginalTOA: Solving single objective optimization problem.
2024/07/11 03:28:18 PM, INFO, mealpy.human_based.TOA.OriginalTOA: >>>Problem: P, Epoch: 1, Current best: -1.59375, Global best: -1.59375, Runtime: 0.54720 seconds
2024/07/11 03:28:18 PM, INFO, mealpy.human_based.TOA.OriginalTOA: >>>Problem: P, Epoch: 2, Current best: -1.78125, Global best: -1.78125, Runtime: 0.53822 seconds
2024/07/11 03:28:19 PM, INFO, mealpy.human_based.TOA.OriginalTOA: >>>Problem: P, Epoch: 3, Current best: -1.78125, Global best: -1.78125, Runtime: 0.54284 seconds
2024/07/11 03:28:19 PM, INFO, mealpy.human_based.TOA.OriginalTOA: >>>Problem: P, Epoch: 4, Current best: -2.03125, Global best: -2.03125, Runtime: 0.63843 seconds
2024/07/11 03:28:20 PM, INFO, mealpy.human_based.TOA.OriginalTOA: >>>Problem: P, Epoch: 5, Current best: -2.03125, Global best: -2.03125, Runtime: 0.62324 seconds
2024/07/11 03:28:21 PM, INFO, mealpy.human_based.TOA.OriginalTOA: >>>Pro

KeyboardInterrupt: 

In [ ]:
from mealpy import FloatVar, WarSO


problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = WarSO.OriginalWarSO(epoch=1000, pop_size=50, rr=0.1)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 03:14:21 PM, INFO, mealpy.human_based.WarSO.OriginalWarSO: Solving single objective optimization problem.
2024/07/11 03:14:21 PM, INFO, mealpy.human_based.WarSO.OriginalWarSO: >>>Problem: P, Epoch: 1, Current best: -1.15625, Global best: -1.15625, Runtime: 0.16161 seconds
2024/07/11 03:14:21 PM, INFO, mealpy.human_based.WarSO.OriginalWarSO: >>>Problem: P, Epoch: 2, Current best: -1.71875, Global best: -1.71875, Runtime: 0.14366 seconds
2024/07/11 03:14:22 PM, INFO, mealpy.human_based.WarSO.OriginalWarSO: >>>Problem: P, Epoch: 3, Current best: -1.71875, Global best: -1.71875, Runtime: 0.13508 seconds
2024/07/11 03:14:22 PM, INFO, mealpy.human_based.WarSO.OriginalWarSO: >>>Problem: P, Epoch: 4, Current best: -1.71875, Global best: -1.71875, Runtime: 0.13905 seconds
2024/07/11 03:14:22 PM, INFO, mealpy.human_based.WarSO.OriginalWarSO: >>>Problem: P, Epoch: 5, Current best: -1.71875, Global best: -1.71875, Runtime: 0.14072 seconds
2024/07/11 03:14:22 PM, INFO, mealpy.human_based

Solution: [ 10.62361515 -16.00955829  12.40577953 -41.59389939   7.13565807
  92.45088721  13.72365524  17.53079149  75.0366794  -32.95864836
   4.33907035   2.31501921 -80.20018439  67.91649165 -17.54170103
  -6.22158991 -92.57516583 -45.46143528 -76.88684     -3.55100291], Fitness: -3.0625
Solution: [ 10.62361515 -16.00955829  12.40577953 -41.59389939   7.13565807
  92.45088721  13.72365524  17.53079149  75.0366794  -32.95864836
   4.33907035   2.31501921 -80.20018439  67.91649165 -17.54170103
  -6.22158991 -92.57516583 -45.46143528 -76.88684     -3.55100291], Fitness: -3.0625


In [39]:
from mealpy import FloatVar, CRO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = CRO.OriginalCRO(epoch=1000, pop_size=50, po = 0.4, Fb = 0.9, Fa = 0.1, Fd = 0.1, Pd = 0.5, GCR = 0.1, gamma_min = 0.02, gamma_max = 0.2, n_trials = 5)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:08:30 PM, INFO, mealpy.evolutionary_based.CRO.OriginalCRO: Solving single objective optimization problem.
2024/07/11 04:08:30 PM, INFO, mealpy.evolutionary_based.CRO.OriginalCRO: >>>Problem: P, Epoch: 1, Current best: -1.65625, Global best: -1.65625, Runtime: 0.06543 seconds
2024/07/11 04:08:30 PM, INFO, mealpy.evolutionary_based.CRO.OriginalCRO: >>>Problem: P, Epoch: 2, Current best: -1.65625, Global best: -1.65625, Runtime: 0.08465 seconds
2024/07/11 04:08:30 PM, INFO, mealpy.evolutionary_based.CRO.OriginalCRO: >>>Problem: P, Epoch: 3, Current best: -1.65625, Global best: -1.65625, Runtime: 0.08665 seconds
2024/07/11 04:08:30 PM, INFO, mealpy.evolutionary_based.CRO.OriginalCRO: >>>Problem: P, Epoch: 4, Current best: -2.875, Global best: -2.875, Runtime: 0.09091 seconds
2024/07/11 04:08:30 PM, INFO, mealpy.evolutionary_based.CRO.OriginalCRO: >>>Problem: P, Epoch: 5, Current best: -2.875, Global best: -2.875, Runtime: 0.10316 seconds
2024/07/11 04:08:30 PM, INFO, mealpy.e

Solution: [ -60.67066768   61.94676293   20.65333427  -53.80012183   62.87510652
   65.01189102 -100.           47.44171732  -65.24222369  -67.50104038
  -92.91317102   23.70956751   39.75931752   48.63401602  -48.63275804
  -17.15700488   76.90476531    1.89188067  -57.85898406  -19.60567467], Fitness: -3.9375
Solution: [ -60.67066768   61.94676293   20.65333427  -53.80012183   62.87510652
   65.01189102 -100.           47.44171732  -65.24222369  -67.50104038
  -92.91317102   23.70956751   39.75931752   48.63401602  -48.63275804
  -17.15700488   76.90476531    1.89188067  -57.85898406  -19.60567467], Fitness: -3.9375


In [41]:
from mealpy import FloatVar, EP


problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = EP.OriginalEP(epoch=1000, pop_size=50, bout_size = 0.05)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")


2024/07/11 04:12:26 PM, INFO, mealpy.evolutionary_based.EP.OriginalEP: Solving single objective optimization problem.
2024/07/11 04:12:26 PM, INFO, mealpy.evolutionary_based.EP.OriginalEP: >>>Problem: P, Epoch: 1, Current best: -1.59375, Global best: -1.59375, Runtime: 0.15858 seconds
2024/07/11 04:12:26 PM, INFO, mealpy.evolutionary_based.EP.OriginalEP: >>>Problem: P, Epoch: 2, Current best: -1.59375, Global best: -1.59375, Runtime: 0.14197 seconds
2024/07/11 04:12:27 PM, INFO, mealpy.evolutionary_based.EP.OriginalEP: >>>Problem: P, Epoch: 3, Current best: -1.59375, Global best: -1.59375, Runtime: 0.15307 seconds
2024/07/11 04:12:27 PM, INFO, mealpy.evolutionary_based.EP.OriginalEP: >>>Problem: P, Epoch: 4, Current best: -1.59375, Global best: -1.59375, Runtime: 0.14193 seconds
2024/07/11 04:12:27 PM, INFO, mealpy.evolutionary_based.EP.OriginalEP: >>>Problem: P, Epoch: 5, Current best: -1.59375, Global best: -1.59375, Runtime: 0.14207 seconds
2024/07/11 04:12:27 PM, INFO, mealpy.evolu

KeyboardInterrupt: 

In [43]:
from mealpy import FloatVar, ES
    
problem_dict = {
    "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = ES.OriginalES(epoch=1000, pop_size=50, lamda = 0.75)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:15:08 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: Solving single objective optimization problem.


2024/07/11 04:15:08 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 1, Current best: -1.03125, Global best: -1.03125, Runtime: 0.10829 seconds
2024/07/11 04:15:08 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 2, Current best: -1.46875, Global best: -1.46875, Runtime: 0.13031 seconds
2024/07/11 04:15:08 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 3, Current best: -1.5625, Global best: -1.5625, Runtime: 0.13073 seconds
2024/07/11 04:15:08 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 4, Current best: -1.5625, Global best: -1.5625, Runtime: 0.12738 seconds
2024/07/11 04:15:08 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 5, Current best: -1.5625, Global best: -1.5625, Runtime: 0.14789 seconds
2024/07/11 04:15:09 PM, INFO, mealpy.evolutionary_based.ES.OriginalES: >>>Problem: P, Epoch: 6, Current best: -1.5625, Global best: -1.5625, Runtime: 0.13685 seconds


KeyboardInterrupt: 

In [45]:
from mealpy import FloatVar, FPA
    
problem_dict = {
    "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    "minmax": "min",
    "obj_func": objective_function
}

model = FPA.OriginalFPA(epoch=1000, pop_size=50, p_s = 0.8, levy_multiplier = 0.2)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:16:58 PM, INFO, mealpy.evolutionary_based.FPA.OriginalFPA: Solving single objective optimization problem.
2024/07/11 04:16:59 PM, INFO, mealpy.evolutionary_based.FPA.OriginalFPA: >>>Problem: P, Epoch: 1, Current best: -1.3125, Global best: -1.3125, Runtime: 0.23089 seconds
2024/07/11 04:16:59 PM, INFO, mealpy.evolutionary_based.FPA.OriginalFPA: >>>Problem: P, Epoch: 2, Current best: -1.3125, Global best: -1.3125, Runtime: 0.20165 seconds
2024/07/11 04:16:59 PM, INFO, mealpy.evolutionary_based.FPA.OriginalFPA: >>>Problem: P, Epoch: 3, Current best: -1.3125, Global best: -1.3125, Runtime: 0.26572 seconds
2024/07/11 04:16:59 PM, INFO, mealpy.evolutionary_based.FPA.OriginalFPA: >>>Problem: P, Epoch: 4, Current best: -1.3125, Global best: -1.3125, Runtime: 0.23638 seconds
2024/07/11 04:17:00 PM, INFO, mealpy.evolutionary_based.FPA.OriginalFPA: >>>Problem: P, Epoch: 5, Current best: -1.53125, Global best: -1.53125, Runtime: 0.29740 seconds
2024/07/11 04:17:00 PM, INFO, mealpy.e

KeyboardInterrupt: 

In [47]:
from mealpy import FloatVar, MA
    
problem_dict = {
    "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    "obj_func": objective_function,
    "minmax": "min",
}

model = MA.OriginalMA(epoch=1000, pop_size=50, pc = 0.85, pm = 0.15, p_local = 0.5, max_local_gens = 10, bits_per_param = 4)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:20:43 PM, INFO, mealpy.evolutionary_based.MA.OriginalMA: Solving single objective optimization problem.
2024/07/11 04:20:45 PM, INFO, mealpy.evolutionary_based.MA.OriginalMA: >>>Problem: P, Epoch: 1, Current best: -1.6875, Global best: -1.6875, Runtime: 1.21772 seconds
2024/07/11 04:20:46 PM, INFO, mealpy.evolutionary_based.MA.OriginalMA: >>>Problem: P, Epoch: 2, Current best: -1.5625, Global best: -1.6875, Runtime: 1.39138 seconds
2024/07/11 04:20:47 PM, INFO, mealpy.evolutionary_based.MA.OriginalMA: >>>Problem: P, Epoch: 3, Current best: -1.5625, Global best: -1.6875, Runtime: 1.01052 seconds
2024/07/11 04:20:48 PM, INFO, mealpy.evolutionary_based.MA.OriginalMA: >>>Problem: P, Epoch: 4, Current best: -1.65625, Global best: -1.6875, Runtime: 1.29257 seconds
2024/07/11 04:20:50 PM, INFO, mealpy.evolutionary_based.MA.OriginalMA: >>>Problem: P, Epoch: 5, Current best: -1.75, Global best: -1.75, Runtime: 1.59834 seconds
2024/07/11 04:20:51 PM, INFO, mealpy.evolutionary_based

KeyboardInterrupt: 

In [48]:
from mealpy import FloatVar, SHADE
    
problem_dict = {
    "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
    "obj_func": objective_function,
    "minmax": "min",
}

model = SHADE.OriginalSHADE(epoch=1000, pop_size=50, miu_f = 0.5, miu_cr = 0.5)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/11 04:21:29 PM, INFO, mealpy.evolutionary_based.SHADE.OriginalSHADE: Solving single objective optimization problem.
2024/07/11 04:21:30 PM, INFO, mealpy.evolutionary_based.SHADE.OriginalSHADE: >>>Problem: P, Epoch: 1, Current best: -1.78125, Global best: -1.78125, Runtime: 0.25783 seconds
2024/07/11 04:21:30 PM, INFO, mealpy.evolutionary_based.SHADE.OriginalSHADE: >>>Problem: P, Epoch: 2, Current best: -1.78125, Global best: -1.78125, Runtime: 0.20843 seconds
2024/07/11 04:21:30 PM, INFO, mealpy.evolutionary_based.SHADE.OriginalSHADE: >>>Problem: P, Epoch: 3, Current best: -1.78125, Global best: -1.78125, Runtime: 0.28385 seconds
2024/07/11 04:21:31 PM, INFO, mealpy.evolutionary_based.SHADE.OriginalSHADE: >>>Problem: P, Epoch: 4, Current best: -2.0, Global best: -2.0, Runtime: 0.27672 seconds
2024/07/11 04:21:31 PM, INFO, mealpy.evolutionary_based.SHADE.OriginalSHADE: >>>Problem: P, Epoch: 5, Current best: -2.0, Global best: -2.0, Runtime: 0.21350 seconds
2024/07/11 04:21:31 PM

Solution: [ 3.80662622  2.18983737 -3.34972226 -0.97331439  7.29224697  1.39255289
  2.3154494   0.34274847 -5.17758131 -2.49576556  0.43363097 -4.54473802
  2.17687495 -1.63150159  6.60379615 -6.3537337  -1.50905481 -1.45336164
 -1.75414971 -2.20244568 -0.83711801  8.3599595   1.39115572 -9.10132186
  1.39969745  0.76185063 -1.21079498 -1.4218195  -9.50006156 -1.39924748], Fitness: -3.96875
Solution: [ 3.80662622  2.18983737 -3.34972226 -0.97331439  7.29224697  1.39255289
  2.3154494   0.34274847 -5.17758131 -2.49576556  0.43363097 -4.54473802
  2.17687495 -1.63150159  6.60379615 -6.3537337  -1.50905481 -1.45336164
 -1.75414971 -2.20244568 -0.83711801  8.3599595   1.39115572 -9.10132186
  1.39969745  0.76185063 -1.21079498 -1.4218195  -9.50006156 -1.39924748], Fitness: -3.96875
